## Setup
[Link to question](https://leetcode.com/problems/monthly-transactions-i/?envType=study-plan-v2&envId=top-sql-50)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import date

# Initialize Spark Session
spark = SparkSession.builder.appName("LeetCode1193").getOrCreate()

# Define Schema
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("trans_date", DateType(), True)
])

# Example Data from LeetCode
data = [
    (121, "US", "approved", 1000, date(2018, 12, 18)),
    (122, "US", "declined", 2000, date(2018, 12, 19)),
    (123, "US", "approved", 2000, date(2019, 1, 1)),
    (124, "DE", "approved", 2000, date(2019, 1, 7))
]

# Create DataFrame
transactions_df = spark.createDataFrame(data, schema)

# Show input
transactions_df.show()

+---+-------+--------+------+----------+
| id|country|   state|amount|trans_date|
+---+-------+--------+------+----------+
|121|     US|approved|  1000|2018-12-18|
|122|     US|declined|  2000|2018-12-19|
|123|     US|approved|  2000|2019-01-01|
|124|     DE|approved|  2000|2019-01-07|
+---+-------+--------+------+----------+



## Code

In [15]:
from pyspark.sql import functions as F

transactions_df.groupBy(F.date_format(F.col('trans_date'), 'yyyy-MM').alias('month'), F.col('country')) \
                  .agg(F.count(F.col('trans_date')).alias('trans_count'), \
                        sum(F.when(F.col('state') == 'approved', 1).otherwise(0)).alias('approved_count'), \
                        sum(F.col('amount')).alias('trans_total_amount'), \
                        sum(F.when(F.col('state') == 'approved', F.col('amount')).otherwise(0)).alias('approved_total_amount ')
                       ).show()

+-------+-------+-----------+--------------+------------------+----------------------+
|  month|country|trans_count|approved_count|trans_total_amount|approved_total_amount |
+-------+-------+-----------+--------------+------------------+----------------------+
|2018-12|     US|          2|             1|              3000|                  1000|
|2019-01|     US|          1|             1|              2000|                  2000|
|2019-01|     DE|          1|             1|              2000|                  2000|
+-------+-------+-----------+--------------+------------------+----------------------+

